# Mask-only control

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os
import pickle

import cv2
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import f1_score, recall_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

ROOT = "/content/drive/MyDrive/TA_Journal"
WORK = {"BUSI": f"{ROOT}/work", "BUS-UCLM": f"{ROOT}/work_uclm"}
WEIGHTS = [1, 1, 2, 1]
SEED = 42


def load(path):
    with open(path, "rb") as f:
        return pickle.load(f)

## Shape descriptors

In [ ]:
def describe(mask):
    m = (mask > 0).astype(np.uint8)
    if m.sum() == 0:
        return [1, 0, 0, 0, 0, 0, 0, 0]
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    c = max(contours, key=cv2.contourArea)
    area = max(cv2.contourArea(c), 1.0)
    hull = max(cv2.contourArea(cv2.convexHull(c)), 1.0)
    _, _, w, h = cv2.boundingRect(c)
    ecc = 0.0
    if len(c) >= 5:
        _, axes, _ = cv2.fitEllipse(c)
        major, minor = max(axes), max(min(axes), 1e-6)
        ecc = np.sqrt(1 - (minor / major) ** 2)
    return [0, m.mean(), len(contours), area / hull, area / (w * h), ecc,
            cv2.arcLength(c, True) ** 2 / (4 * np.pi * area), w / h]

## Ensemble on the outer folds

In [ ]:
def classifiers():
    return [DecisionTreeClassifier(random_state=0),
            KNeighborsClassifier(5),
            RandomForestClassifier(100, random_state=0, n_jobs=-1),
            HistGradientBoostingClassifier(max_iter=100, max_depth=3, max_bins=64,
                                           early_stopping=False, random_state=0)]


def evaluate(features_per_fold, y, folds):
    pred, fold_f1 = np.zeros(len(y), int), []
    for (tr, te), F in zip(folds, features_per_fold):
        scaler = StandardScaler().fit(F[tr])
        prob = np.stack([c.fit(scaler.transform(F[tr]), y[tr]).predict_proba(scaler.transform(F[te]))
                         for c in classifiers()])
        pred[te] = np.tensordot(np.asarray(WEIGHTS) / sum(WEIGHTS), prob, 1).argmax(-1)
        fold_f1.append(f1_score(y[te], pred[te], average="macro"))
    rng = np.random.default_rng(SEED)
    boot = [f1_score(y[i], pred[i], average="macro")
            for i in (rng.integers(0, len(y), len(y)) for _ in range(1000))]
    recall = recall_score(y, pred, average=None)
    return dict(macro_f1=f1_score(y, pred, average="macro"),
                ci_low=np.percentile(boot, 2.5), ci_high=np.percentile(boot, 97.5),
                fold_mean=np.mean(fold_f1), fold_sd=np.std(fold_f1, ddof=1),
                recall_benign=recall[0], recall_malignant=recall[1], recall_normal=recall[2])

## Run

In [ ]:
rows = []
for dataset, work in WORK.items():
    d = load(f"{work}/data.pkl")
    y, folds = d["y"], d["folds"]
    expert = np.array([describe(m) for m in d["mask"]], float)
    rows.append(dict(dataset=dataset, masks="expert", **evaluate([expert] * len(folds), y, folds)))
    predicted = [np.array([describe(m) for m in load(f"{work}/seg/fold{k}.pkl")], float)
                 for k in range(len(folds))]
    rows.append(dict(dataset=dataset, masks="predicted", **evaluate(predicted, y, folds)))

result = pd.DataFrame(rows).round(3)
result.to_csv(f"{ROOT}/mask_only.csv", index=False)
result

,dataset,masks,macro_f1,ci_low,ci_high,fold_mean,fold_sd,recall_benign,recall_malignant,recall_normal
0,BUSI,expert,0.974,0.962,0.985,0.974,0.006,0.979,0.938,1.000
1,BUSI,predicted,0.668,0.632,0.701,0.639,0.097,0.771,0.524,0.729
2,BUS-UCLM,expert,0.798,0.756,0.836,0.767,0.081,0.828,0.556,1.000
3,BUS-UCLM,predicted,0.557,0.515,0.598,0.506,0.145,0.437,0.244,0.947
